# Uterine MRI pipeline on the UMD dataset (Colab / Kaggle)
Runs the nnU-Net v2 ResEnc-M pipeline end-to-end on the public UMD dataset (Pan et al., 2024; distributed as `UMD.zip` on figshare).

**Before you start**
1. Runtime: GPU (Colab: Runtime > Change runtime type > T4 or better. Kaggle: Settings > Accelerator).
2. Have `uterine_mri_tool.py` ready (upload it in step 2).
3. Get `UMD.zip` (figshare, search "UMD.zip Pan figshare" or use the link in the Scientific Data paper's Data Availability section).

**Compute reality check:** the paper trains 5 folds x 1000 epochs on a V100. That is not realistic in a free notebook session. This notebook defaults to 1 fold x 100 epochs (`nnUNetTrainer_100epochs`). Scale `EPOCH_TRAINER` and `FOLDS` up if you have the time.

## 1. Setup

In [ ]:
import os, sys, glob, json, random, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/work'
    PERSIST = '/content/drive/MyDrive/uterine_nnunet'   # survives session resets
else:  # Kaggle
    WORK = '/kaggle/working/work'
    PERSIST = '/kaggle/working/persist'
for p in (WORK, PERSIST): os.makedirs(p, exist_ok=True)

# raw + preprocessed on fast local disk, results (checkpoints) on persistent storage
os.environ['nnUNet_raw'] = f'{WORK}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{WORK}/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{PERSIST}/nnUNet_results'
for k in ('nnUNet_raw','nnUNet_preprocessed','nnUNet_results'): os.makedirs(os.environ[k], exist_ok=True)

In [ ]:
!pip -q install nnunetv2 nibabel fire SimpleITK matplotlib
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Get the code and the data

In [ ]:
# Colab: upload uterine_mri_tool.py (or copy it from Drive). Kaggle: add it as a dataset / upload to /kaggle/working.
if IN_COLAB:
    from google.colab import files
    if not os.path.exists('uterine_mri_tool.py'):
        files.upload()
assert os.path.exists('uterine_mri_tool.py'), 'put uterine_mri_tool.py in the working directory'

# Make the number of ensemble folds configurable (the original hardcodes all 5)
src = open('uterine_mri_tool.py').read()
if 'FOLDS =' not in src:
    src = src.replace('CONFIG = "3d_fullres"', 'CONFIG = "3d_fullres"\nFOLDS = (0, 1, 2, 3, 4)')
    src = src.replace('use_folds=(0, 1, 2, 3, 4)', 'use_folds=FOLDS')
    open('uterine_mri_tool.py', 'w').write(src)

In [ ]:
# --- Option A (Colab): download from figshare. Paste the direct download URL you get from the figshare page. ---
UMD_URL = ''   # e.g. 'https://figshare.com/ndownloader/files/<id>'
if UMD_URL:
    !wget -q -O {WORK}/UMD.zip "{UMD_URL}"
# --- Option B (Kaggle): add UMD.zip as an input dataset, then point to it: ---
# shutil.copy('/kaggle/input/<your-dataset>/UMD.zip', f'{WORK}/UMD.zip')
# --- Option C: upload UMD.zip to Drive and copy it: ---
# shutil.copy('/content/drive/MyDrive/UMD.zip', f'{WORK}/UMD.zip')

!unzip -q -o {WORK}/UMD.zip -d {WORK}/UMD_raw

## 3. Inspect the folder structure

(Skip steps 3 and 4 if UMD dataset has already been unzipped and cleaned)

In [ ]:
from collections import Counter
files_ = [p for p in Path(f'{WORK}/UMD_raw').rglob('*') if p.is_file()]
print(len(files_), 'files')
print(Counter(''.join(p.suffixes) for p in files_).most_common())
for p in files_[:25]: print(p.relative_to(f'{WORK}/UMD_raw'))

## 4. Standardize into paired NIfTI volumes
Edit `find_pairs()` to match what step 3 printed. The default assumes one image file and one label file per patient with similar names (e.g. `case/img.nii.gz` + `case/label.nii.gz`, or `images/x.nii.gz` + `labels/x.nii.gz`). If the data is DICOM or an image-slice format, `SimpleITK` below reads it (for a DICOM series pass the folder to `read_any`).

In [ ]:
import numpy as np, SimpleITK as sitk

def read_any(path):
    path = Path(path)
    if path.is_dir():   # DICOM series folder
        r = sitk.ImageSeriesReader()
        r.SetFileNames(r.GetGDCMSeriesFileNames(str(path)))
        return r.Execute()
    return sitk.ReadImage(str(path))

def find_pairs(root):
    """Return list of (case_id, image_path, label_path). ADAPT TO THE ACTUAL LAYOUT."""
    root = Path(root)
    pairs = []
    imgs = [p for p in root.rglob('*.nii*') if not any(k in p.name.lower() for k in ('label','mask','seg','annot'))]
    for im in sorted(imgs):
        # look for a label file in the same folder, or with a matching stem elsewhere
        cands = [p for p in root.rglob('*.nii*') if p != im and
                 (p.parent == im.parent or im.name.split('.')[0] in p.name) and
                 any(k in str(p).lower() for k in ('label','mask','seg','annot'))]
        if cands:
            pairs.append((f'{len(pairs):03d}', im, cands[0]))
    return pairs

pairs = find_pairs(f'{WORK}/UMD_raw')
print(len(pairs), 'pairs (expect 300)')
for p in pairs[:3]: print(p)

In [ ]:
# What integer values do the label maps contain? Expect 0 + four structures.
vals = Counter()
for _, _, lp in pairs[:20]:
    vals.update(np.unique(sitk.GetArrayFromImage(read_any(lp))).tolist())
print(vals)

In [ ]:
# Map the dataset's label values to this project's convention:
#   0 background, 1 uterine wall, 2 uterine cavity, 3 myoma, 4 Nabothian cyst
# Fill in after checking the UMD documentation / viewing a few cases (e.g. in ITK-SNAP or 3D Slicer).
LABEL_REMAP = {}      # e.g. {0:0, 1:1, 2:2, 3:3, 4:4}
STD = Path(f'{WORK}/std'); (STD/'images').mkdir(parents=True, exist_ok=True); (STD/'labels').mkdir(exist_ok=True)

for cid, ip, lp in pairs:
    img = read_any(ip)
    lab = read_any(lp)
    arr = sitk.GetArrayFromImage(lab).astype(np.uint8)
    if LABEL_REMAP:
        out = np.zeros_like(arr)
        for src_v, dst_v in LABEL_REMAP.items(): out[arr == src_v] = dst_v
        arr = out
    lab2 = sitk.GetImageFromArray(arr); lab2.CopyInformation(img)   # force identical geometry
    sitk.WriteImage(sitk.Cast(img, sitk.sitkFloat32), str(STD/'images'/f'{cid}.nii.gz'))
    sitk.WriteImage(lab2, str(STD/'labels'/f'{cid}.nii.gz'))
print('done')

## 5. Build the nnU-Net dataset (240 train / 60 test) and preprocess

In [ ]:
import uterine_mri_tool as u
ids = sorted(p.name.replace('.nii.gz','') for p in (STD/'images').glob('*.nii.gz'))
random.seed(0)
test_ids = random.sample(ids, 60) if len(ids) >= 300 else random.sample(ids, max(1, len(ids)//5))
json.dump(test_ids, open(f'{PERSIST}/test_ids.json','w'))
u.prepare_dataset(str(STD/'images'), str(STD/'labels'), test_ids=test_ids)

In [ ]:
# ResEnc-M planner, only the 3d_fullres config (skips 2d/lowres to save time)
!nnUNetv2_plan_and_preprocess -d 501 -pl nnUNetPlannerResEncM -c 3d_fullres --verify_dataset_integrity

## 6. Train
`nnUNetTrainer_100epochs` is a built-in nnU-Net variant. Use `nnUNetTrainer` (default) for the full 1000 epochs. Check the time of the first few epochs in the log and extrapolate before committing. `--c` resumes from the last checkpoint after a session reset, since results live on persistent storage.

In [ ]:
EPOCH_TRAINER = 'nnUNetTrainer_100epochs'   # 'nnUNetTrainer' = 1000 epochs as in the paper
FOLD = 0
!nnUNetv2_train 501 3d_fullres {FOLD} -p nnUNetResEncUNetMPlans -tr {EPOCH_TRAINER} --c

In [ ]:
# Optional: repeat for more folds (each is a separate multi-hour run)
# for f in (1,2,3,4): !nnUNetv2_train 501 3d_fullres {f} -p nnUNetResEncUNetMPlans -tr {EPOCH_TRAINER} --c

## 7. Evaluate on the 60 held-out cases (Dice / IoU)

In [ ]:
RAW = Path(os.environ['nnUNet_raw'])/'Dataset501_UterusMRI'
!nnUNetv2_predict -i {RAW}/imagesTs -o {WORK}/preds_test -d 501 -c 3d_fullres -p nnUNetResEncUNetMPlans -tr {EPOCH_TRAINER} -f {FOLD}
u.evaluate(f'{WORK}/preds_test', str(RAW/'labelsTs'), out_json=f'{PERSIST}/test_metrics.json')

## 8. Run the full tool on one scan and view the report

In [ ]:
u.TRAINER = EPOCH_TRAINER
u.FOLDS = (FOLD,)              # use all five once you have trained them
case = sorted((RAW/'imagesTs').glob('*_0000.nii.gz'))[0]
res = u.run(input=str(case), output_dir=f'{WORK}/report_demo')
from IPython.display import HTML, display
display(HTML(open(f'{WORK}/report_demo/report.html').read()))

In [ ]:
# Save the report and metrics somewhere persistent
shutil.copytree(f'{WORK}/report_demo', f'{PERSIST}/report_demo', dirs_exist_ok=True)